# **Explore Hessen Soccer**

In [1]:
from utils import write_scraped_df, concat_scrape, manual_geocode
import pandas as pd
import osmnx as ox
pd.set_option('display.max_rows', 200)

# define coordinate reference system
CRS = 5243

See ``wiki_scraper`` from utils to see how each table was scraped.

In [2]:
liga_dict = {
'verbandsliga_nord': 'https://www.fussball.de/spieltag/vl-grnord-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318HV9REO00000AVS5489BUVV628VP4-G#!/section/matches',
'verbandsliga_mitte': 'https://www.fussball.de/spieltag/vl-grmitte-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318I3K77C00000BVS5489BUVV628VP4-G#!/section/matches',
'verbandsliga_sued': 'https://www.fussball.de/spieltag/vl-grsued-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318I2SMPG00000CVS5489BUVV628VP4-G#!/section/matches',
'hessenliga': 'https://www.fussball.de/spieltag/lotto-hessenliga-hessen-hessenliga-herren-saison2627-hessen/-/staffel/0318HOM0R000000BVS5489BUVV628VP4-G#!/section/matches',
'regionalliga_suedwest': 'https://www.fussball.de/spieltag/regionalliga-suedwest-deutschland-regionalliga-suedwest-herren-saison2627-deutschland/-/staffel/031D0NRQ3O000004VS5489BUVUR5FS5A-G#!/section/matches',
'3_bundesliga': 'https://www.fussball.de/spieltag/3liga-deutschland-3liga-herren-saison2627-deutschland/-/staffel/031BH6P20G00000HVS5489BUVUR5FS5A-G#!/section/matches',
'2_bundesliga': 'https://www.fussball.de/spieltag/2bundesliga-deutschland-2bundesliga-herren-saison2627-deutschland/-/staffel/031BG5ESP800000TVS5489BUVUR5FS5A-G#!/section/matches',
'bundesliga': 'https://www.fussball.de/spieltag/bundesliga-deutschland-bundesliga-herren-saison2627-deutschland/-/staffel/031BFREJC800001BVS5489BUVUR5FS5A-G#!/section/matches'
}

# write scraped tables from URL
write_scraped_df(d=liga_dict)
bl_to_vl = concat_scrape(liga_dict)
bl_to_vl.to_csv('bl_to_vl.csv', index=False)
bl_to_vl.info()

<class 'pandas.DataFrame'>
Index: 143 entries, 0 to 17
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pl.platz            0 non-null      float64
 1   pl.platz.1          143 non-null    float64
 2   mannschaft          143 non-null    str    
 3   sp.spiele           143 non-null    int64  
 4   g                   143 non-null    int64  
 5   u                   143 non-null    int64  
 6   v                   143 non-null    int64  
 7   torv.torverhältnis  143 non-null    str    
 8   tordifferenz        143 non-null    int64  
 9   pkt.punkte          143 non-null    int64  
 10  liga                143 non-null    str    
dtypes: float64(2), int64(6), str(3)
memory usage: 13.4 KB


In [3]:
bl_to_vl

,pl.platz,pl.platz.1,mannschaft,sp.spiele,g,u,v,torv.torverhältnis,tordifferenz,pkt.punkte,liga
0,NaN,1.0,SG Klei./​Hun./​Doh.,18,12,2,4,42 : 26,16,38,verbandsliga_nord
1,NaN,2.0,SV Steinbach,17,11,3,3,45 : 22,23,36,verbandsliga_nord
2,NaN,3.0,Tuspo Rengershausen,18,10,5,3,52 : 28,24,35,verbandsliga_nord
3,NaN,4.0,SV Weidenhausen,17,11,1,5,26 : 19,7,34,verbandsliga_nord
4,NaN,5.0,SG Bad Soden,18,11,1,6,47 : 22,25,33,verbandsliga_nord
5,NaN,6.0,FSV Wolfhagen,18,10,2,6,29 : 17,12,32,verbandsliga_nord
6,NaN,7.0,OSC Vellmar,18,9,2,7,44 : 38,6,29,verbandsliga_nord
7,NaN,8.0,TSV 1900 Wabern,18,7,5,6,30 : 25,5,26,verbandsliga_nord
8,NaN,9.0,SV Buchonia Flieden,18,6,6,6,34 : 33,1,24,verbandsliga_nord
9,NaN,10.0,FSV Dörnberg,18,7,3,8,24 : 37,-13,24,verbandsliga_nord


In [4]:
gdf_list = []
geocode_manually = []
for i, mannschaft in enumerate(bl_to_vl['mannschaft']):
    try:
        geocoded_mannschaft = ox.geocode_to_gdf(mannschaft)
        geocoded_mannschaft['mannschaft'] = mannschaft
        # geocoded_mannschaft['liga'] = bl_to_vl.iloc[i]['liga'] # add liga w/ merge
        gdf_list.append(geocoded_mannschaft) # add the club name and league to the record for merging
    except:
        print(mannschaft)
        geocode_manually.append(mannschaft)
        continue

print('Geocoded Teams',len(gdf_list))
print('Need Geocoding',len(bl_to_vl))

SG Klei./​Hun./​Doh.
FSV Wolfhagen
OSC Vellmar
FSV Dörnberg
SG Barockstadt Fulda-Lehnerz II
SG Calden/​Meimbressen
FC Britannia Eichenzell
Tuspo Grebenstein
SG Bronnzell
Lichtenauer FV
TSV Steinbach Haiger II
SG Walluf
FV Biebrich 02
FC Ederbergland
SF/​BG Marburg
SV Zeilsheim
Türkischer SV Wiesbaden
SSC Juno Burg
SG Obbornh./​Bellersh.
TSF Heuchelheim
TuRa Niederhöchstadt
SV Pars Neu-Isenburg
DJK Spfr. Bad Homburg
SV Bosnien/​Herzegowina Ffm
Spfrd. Seligenstadt
VfR Fehlheim
SG Bornheim/​GW Ffm
SV Rot-Weiss Walldorf II
Kickers Offenbach II
SG Rot-Weiss Ffm.
1. Hanauer FC
Spvgg.Neu-Isenburg
TSV Eintr. Stadtallendorf
Türk Gücü Friedberg
SV Darmstadt 98 U21
FC Bayern Alzenau
FC Turabdin-Babylon
FSV Fernwald
Hünfelder SV
SC 1960 Hanau
SV Rot-Weiss Walldorf
SV Hummetroth
KSV Hessen Kassel II
SGV Heilbronn-Freiberg
VfR Mannheim
SV Stuttgarter Kickers
Kickers Offenbach
TSV Steinbach Haiger
SC Freiburg II
Eintracht Frankfurt II
SSV Ulm 1846 Fußball
1. FSV Mainz 05 II
1. FC Kaiserslautern II
MS

In [5]:
manually_geocoded = {}

In [6]:
geocode_manually[8:]

['SG Bronnzell',
 'Lichtenauer FV',
 'TSV Steinbach Haiger II',
 'SG Walluf',
 'FV Biebrich 02',
 'FC Ederbergland',
 'SF/\u200bBG Marburg',
 'SV Zeilsheim',
 'Türkischer SV Wiesbaden',
 'SSC Juno Burg',
 'SG Obbornh./\u200bBellersh.',
 'TSF Heuchelheim',
 'TuRa Niederhöchstadt',
 'SV Pars Neu-Isenburg',
 'DJK Spfr. Bad Homburg',
 'SV Bosnien/\u200bHerzegowina Ffm',
 'Spfrd. Seligenstadt',
 'VfR Fehlheim',
 'SG Bornheim/\u200bGW Ffm',
 'SV Rot-Weiss Walldorf II',
 'Kickers Offenbach II',
 'SG Rot-Weiss Ffm.',
 '1. Hanauer FC',
 'Spvgg.Neu-Isenburg',
 'TSV Eintr. Stadtallendorf',
 'Türk Gücü Friedberg',
 'SV Darmstadt 98 U21',
 'FC Bayern Alzenau',
 'FC Turabdin-Babylon',
 'FSV Fernwald',
 'Hünfelder SV',
 'SC 1960 Hanau',
 'SV Rot-Weiss Walldorf',
 'SV Hummetroth',
 'KSV Hessen Kassel II',
 'SGV Heilbronn-Freiberg',
 'VfR Mannheim',
 'SV Stuttgarter Kickers',
 'Kickers Offenbach',
 'TSV Steinbach Haiger',
 'SC Freiburg II',
 'Eintracht Frankfurt II',
 'SSV Ulm 1846 Fußball',
 '1. FSV M

In [ ]:
for c in geocode_manually[12:]:
    print(c)
    manually_geocoded = manual_geocode(d=manually_geocoded, file='new_manually_geocoded.csv')

SG Walluf
FV Biebrich 02
try again, or be more specific


TypeError: Nominatim did not geocode query 'Diltheystraße, 65203 Biebrich' to a geometry of type (Multi)Polygon.

In [ ]:
manually_geocoded["SG Rot-Weiss Ffm."] = ox.geocode_to_gdf('60461 Frankfurt am Main')
pd.concat([v for k, v in manually_geocoded.items()]).to_csv('manually_geocoded.csv', index=False)
manually_geocoded

{'SG Klei./Hun./Doh.':                                             geometry  bbox_west  bbox_south  \
 0  POLYGON ((9.77793 51.32669, 9.77806 51.3266, 9...   9.777935   51.326601   
 
    bbox_east  bbox_north   place_id osm_type     osm_id        lat       lon  \
 0   9.778236   51.326788  140187399      way  445922500  51.326695  9.778085   
 
       class   type  place_rank  importance addresstype name  \
 0  building  house          30    0.000055    building        
 
                                         display_name  
 0  16, Waldweg, Kleinalmerode, Witzenhausen, Werr...  ,
 'FSV Wolfhagen':                                             geometry  bbox_west  bbox_south  \
 0  POLYGON ((9.48745 51.29332, 9.48746 51.29328, ...   9.487446   51.293215   
 
    bbox_east  bbox_north   place_id osm_type      osm_id        lat       lon  \
 0   9.487808   51.293442  139481251      way  1448662642  51.293331  9.487628   
 
      class      type  place_rank  importance addresstype       

In [ ]:
import pandas as pd
pd.read_csv('manually_geocoded.csv')

,geometry,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name
0,"POLYGON ((9.7779347 51.3266852, 9.7780594 51.3...",9.777935,51.326601,9.778236,51.326788,140187399,way,445922500,51.326695,9.778085,building,house,30,0.000055,building,NaN,"16, Waldweg, Kleinalmerode, Witzenhausen, Werr..."
1,"POLYGON ((9.4874459 51.2933168, 9.4874605 51.2...",9.487446,51.293215,9.487808,51.293442,139481251,way,1448662642,51.293331,9.487628,highway,tertiary,26,0.053406,tertiary,Am Sportzentrum,"Am Sportzentrum, Südstadt, Kassel, Hesse, 3412..."
2,"POLYGON ((9.4281802 51.3746721, 9.4283032 51.3...",9.428180,51.342456,9.500924,51.382031,139608986,relation,532030,51.362249,9.469233,boundary,administrative,16,0.442743,town,Vellmar,"Vellmar, Landkreis Kassel, Hesse, 34246, Germany"
3,"POLYGON ((9.3502404 51.3416284, 9.3503077 51.3...",9.350240,51.341468,9.350470,51.341655,139534024,way,556265640,51.341562,9.350355,building,residential,30,0.000050,building,NaN,"8, An der Esche, Dörnberg, Habichtswald, Landk..."
4,"POLYGON ((9.6930853 50.5701319, 9.693132 50.57...",9.693085,50.570034,9.693530,50.570273,138212583,way,819170010,50.570150,9.693278,building,yes,30,0.000064,building,Vereinshaus SG Barockstadt Fulda-Lehnerz,"Vereinshaus SG Barockstadt Fulda-Lehnerz, 16, ..."
5,"POLYGON ((9.264784 51.4280647, 9.2650031 51.42...",9.264784,51.365981,9.444308,51.453451,139663943,relation,455491,51.396557,9.364220,boundary,administrative,16,0.430147,municipality,Calden,"Calden, Landkreis Kassel, Hesse, 34379, Germany"
6,"POLYGON ((9.6461388 50.4803081, 9.6466238 50.4...",9.646139,50.413083,9.790390,50.511298,137597497,relation,454872,50.484723,9.705010,boundary,administrative,16,0.416605,municipality,Eichenzell,"Eichenzell, Landkreis Fulda, Hesse, 36124, Ger..."
7,"POLYGON ((9.4135097 51.4471089, 9.4135424 51.4...",9.413510,51.447013,9.413860,51.447142,139471455,way,194600344,51.447077,9.413686,building,retail,30,0.000053,building,NaN,"26, Marktstraße, Kressenbrunnen, Grebenstein, ..."
8,"POLYGON ((9.6848928 50.5184574, 9.6849349 50.5...",9.684893,50.518365,9.685123,50.518507,138123133,way,444994622,50.518426,9.685016,building,yes,30,0.000064,building,NaN,"2, Wartburgstraße, Bronnzell, Fulda, Landkreis..."
9,"POLYGON ((9.7196457 51.1919574, 9.7197072 51.1...",9.719646,51.191901,9.719862,51.192041,139601348,way,680009600,51.191986,9.719754,building,yes,30,0.000053,building,NaN,"21, Thüringer Straße, Hessisch Lichtenau, Werr..."


In [ ]:
man_geo_df = pd.read_csv('manually_geocoded.csv')
print(len(man_geo_df))

77


In [ ]:
len(manually_geocoded.keys())

77

In [ ]:
import numpy as np
np.where(man_geo_df['place_id'] == 138123133)[0][0]

np.int64(8)

In [ ]:
for k, v in manually_geocoded.items():
    print(k)
    # man_geo_df[np.where(man_geo_df['place_id'] == v['place_id'][0])[0][0], 'mannschaft'] = k

SG Klei./Hun./Doh.
FSV Wolfhagen
OSC Vellmar
FSV Dörnberg
SG Barockstadt Fulda-Lehnerz II
SG Calden/​Meimbressen
FC Britannia Eichenzell
Tuspo Grebenstein
SG Bronnzell
Lichtenauer FV
TSV Steinbach Haiger II
TSV Steinbach Haiger
SG Walluf
FV Biebrich 02
FC Ederbergland
SF/​BG Marburg
SV Zeilsheim
Türkischer SV Wiesbaden
SSC Juno Burg
SG Obbornh./​Bellersh.
TSF Heuchelheim
TuRa Niederhöchstadt
SV Pars Neu-Isenburg
DJK Spfr. Bad Homburg
SV Bosnien/​Herzegowina Ffm
Spfrd. Seligenstadt
VfR Fehlheim
SG Bornheim/​GW Ffm
SV Rot-Weiss Walldorf II
Kickers Offenbach II
Kickers Offenbach
SG Rot-Weiss Ffm
1. Hanauer FC
Spvgg.Neu-Isenburg
TSV Eintr. Stadtallendorf
Türk Gücü Friedberg
SV Darmstadt 98 U21
SV Darmstadt
FC Bayern Alzenau
FC Turabdin-Babylon
FSV Fernwald
Hünfelder SV
SC 1960 Hanau
SV Rot-Weiss Walldorf
SV Hummetroth
KSV Hessen Kassel II
SGV Heilbronn-Freiberg
VfR Mannheim
SV Stuttgarter Kickers
SC Freiburg II
Eintracht Frankfurt II
SSV Ulm 1846 Fußball
1. FSV Mainz 05 II
1. FC Kaiserslau

In [ ]:
man_geo_df

,geometry,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name
0,"POLYGON ((9.7779347 51.3266852, 9.7780594 51.3...",9.777935,51.326601,9.778236,51.326788,140187399,way,445922500,51.326695,9.778085,building,house,30,0.000055,building,NaN,"16, Waldweg, Kleinalmerode, Witzenhausen, Werr..."
1,"POLYGON ((9.4874459 51.2933168, 9.4874605 51.2...",9.487446,51.293215,9.487808,51.293442,139481251,way,1448662642,51.293331,9.487628,highway,tertiary,26,0.053406,tertiary,Am Sportzentrum,"Am Sportzentrum, Südstadt, Kassel, Hesse, 3412..."
2,"POLYGON ((9.4281802 51.3746721, 9.4283032 51.3...",9.428180,51.342456,9.500924,51.382031,139608986,relation,532030,51.362249,9.469233,boundary,administrative,16,0.442743,town,Vellmar,"Vellmar, Landkreis Kassel, Hesse, 34246, Germany"
3,"POLYGON ((9.3502404 51.3416284, 9.3503077 51.3...",9.350240,51.341468,9.350470,51.341655,139534024,way,556265640,51.341562,9.350355,building,residential,30,0.000050,building,NaN,"8, An der Esche, Dörnberg, Habichtswald, Landk..."
4,"POLYGON ((9.6930853 50.5701319, 9.693132 50.57...",9.693085,50.570034,9.693530,50.570273,138212583,way,819170010,50.570150,9.693278,building,yes,30,0.000064,building,Vereinshaus SG Barockstadt Fulda-Lehnerz,"Vereinshaus SG Barockstadt Fulda-Lehnerz, 16, ..."
5,"POLYGON ((9.264784 51.4280647, 9.2650031 51.42...",9.264784,51.365981,9.444308,51.453451,139663943,relation,455491,51.396557,9.364220,boundary,administrative,16,0.430147,municipality,Calden,"Calden, Landkreis Kassel, Hesse, 34379, Germany"
6,"POLYGON ((9.6461388 50.4803081, 9.6466238 50.4...",9.646139,50.413083,9.790390,50.511298,137597497,relation,454872,50.484723,9.705010,boundary,administrative,16,0.416605,municipality,Eichenzell,"Eichenzell, Landkreis Fulda, Hesse, 36124, Ger..."
7,"POLYGON ((9.4135097 51.4471089, 9.4135424 51.4...",9.413510,51.447013,9.413860,51.447142,139471455,way,194600344,51.447077,9.413686,building,retail,30,0.000053,building,NaN,"26, Marktstraße, Kressenbrunnen, Grebenstein, ..."
8,"POLYGON ((9.6848928 50.5184574, 9.6849349 50.5...",9.684893,50.518365,9.685123,50.518507,138123133,way,444994622,50.518426,9.685016,building,yes,30,0.000064,building,NaN,"2, Wartburgstraße, Bronnzell, Fulda, Landkreis..."
9,"POLYGON ((9.7196457 51.1919574, 9.7197072 51.1...",9.719646,51.191901,9.719862,51.192041,139601348,way,680009600,51.191986,9.719754,building,yes,30,0.000053,building,NaN,"21, Thüringer Straße, Hessisch Lichtenau, Werr..."


In [ ]:
gdf = pd.concat(gdf_list)
gdf = gdf.to_crs(CRS)
gdf.head(1)

,geometry,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name,mannschaft
0,"POLYGON ((-64305.066 -109941.467, -64220.887 -...",9.602263,50.006636,9.604056,50.007771,136991363,way,26572916,50.007217,9.603158,leisure,pitch,30,0.000055,leisure,SV Steinbach,"SV Steinbach, Lohrer Straße, Steinbach, Lohr a...",SV Steinbach


In [ ]:
for man in geocode_manually:
    if man not in manually_geocoded.keys():
        print(man)

SG Klei./​Hun./​Doh.
SG Rot-Weiss Ffm.


In [ ]:
gdf['mannschaft']

0                    SV Steinbach
0             Tuspo Rengershausen
0                 SV Weidenhausen
0                    SG Bad Soden
0                 TSV 1900 Wabern
0             SV Buchonia Flieden
0               1.FC Schwalmstadt
0                      TuS Hornau
0                     TSG Wieseck
0                    FC Waldbrunn
0                 TUS Dietkirchen
0             VFB Unterliederbach
0                   SC Waldgirmes
0             Germania Ober-Roden
0                    VfB Ginsheim
0          SKV Rot-Weiß Darmstadt
0                      FC Kalbach
0                   SC Dortelweil
0                  VFR Groß-Gerau
0            SV Unter-Flockenbach
0             SV Rot-Weiß Hadamar
0                   FC Eddersheim
0                VfB 1905 Marburg
0                   CSC 03 Kassel
0                    KSV Baunatal
0                      FC Giessen
0                       VfR Aalen
0              SV Eintracht-Trier
0                   FC 08 Homburg
0             

In [ ]:
gdf['geometry'] = gdf['geometry'].centroid
gdf.info()

In [ ]:
gdf.explore(tooltip=False, popup=True)

In [ ]:
# write geocoded df
# gdf.to_file('data/hessen_fussball_gdf.geojson', driver='GeoJSON')